In [5]:
from dh_tool import Sheets, load, save
from dh_tool.common import *
column_with_config = {
        "Model": 20,
        "CommentCode": 16, 
        "Comment": 80,
       '과정_중심_서술_(process_description)_reason':60,
       '탐구_주제의_학술성_(academic_value)_reason':60,
       '비판적_분석력_(critical_analysis)_reason':60,
       '결과_해석의_발전성_(result_development)_reason':60,
    '탐구_동기(exploration_motivation)_reason':60
    }

In [6]:
def sheet_format(sheet, column_with_config):
    sheet.set_column_width(**column_with_config)
    sheet.freeze_first_row()
    sheet.enable_autowrap()
    # sheet.
    return sheet

def get_cell_address(df, column_name):
    """
    DataFrame에서 특정 컬럼의 각 행에 해당하는 셀 주소를 계산
    - df: 대상 DataFrame
    - column_name: 셀 주소를 계산할 컬럼 이름
    - 반환값: {값: 셀 주소} 형태의 딕셔너리
    """
    addresses = {}
    col_idx = df.columns.get_loc(column_name)  # 컬럼 인덱스 (0부터 시작)
    for i, value in enumerate(df[column_name], start=2):  # 2부터 시작 (엑셀 헤더 제외)
        cell_address = f"{chr(65 + col_idx)}{i}"  # 컬럼을 알파벳으로 변환
        addresses[value] = cell_address
    return addresses

for path in Path('../result/2025-01-17').glob('*'):
    print(path)
    meta_df = load(path / 'meta.xlsx')
    result_df = load(path / 'result.xlsx')
    result_columns = [col for col in result_df.columns if col.endswith('_result')]
    result_df['4개항목_좋음개수_result'] = result_df[result_columns].apply(lambda row: (row == '좋음').sum(), axis=1)
    result_df['4개항목_보통개수_result'] = result_df[result_columns].apply(lambda row: (row == '보통').sum(), axis=1)
    result_counts = result_df.groupby('Model')[result_columns].apply(lambda group: group.apply(pd.Series.value_counts).fillna(0)).reset_index()

    sheet = Sheets(pd.DataFrame(None))
    sheet.create_sheet(
        meta_df, 
        '원본', 
    )
    sheet = sheet_format(sheet, column_with_config)
    sheet.create_sheet(
        result_df, 
        '결과', 
    )
    hyper_link_format = "#'원본'!{}"
    comment_code_cell_address = get_cell_address(meta_df, 'CommentCode')
    hyper_link_list = [hyper_link_format.format(comment_code_cell_address[i]) for i in result_df['CommentCode'].values]
    # break
    sheet.add_hyperlinks_to_column("CommentCode", hyper_link_list,)
    sheet = sheet_format(sheet, column_with_config)
    sheet.create_sheet(result_counts, '결과_개수')
    sheet = sheet_format(sheet, column_with_config)
    sheet.remove_sheet("Sheet1")
    sheet.save(path / '종합.xlsx')



../result/2025-01-17/v02
../result/2025-01-17/v03
../result/2025-01-17/v04
../result/2025-01-17/v05
../result/2025-01-17/v06
../result/2025-01-17/v07
